# 35 — Branch C pilot: does the siblings' common subspace align with the hypernym?

Park et al. (2406.01506) prove hierarchy-as-orthogonality: a child concept
decomposes as parent ⊕ orthogonal residual. Fukui's GDS says the common
subspace of a set of class subspaces is their shared structure. Composing
the two gives the Branch-C hypothesis:

> **The GDS common subspace M of sibling concept frames aligns with their
> WordNet hypernym's frame** — measurably, in canonical angles, far above
> the cross-family null.

If it holds, GDS becomes an *unsupervised taxonomist*: hierarchy induced
from unembedding geometry alone, validated against WordNet, applicable
where WordNet is silent. This pilot tests it on families mined from the
concept universe (hypernym + >= 3 surviving siblings, all passing the
standard concept filters), bf16 Llama, no text generation — pure geometry.

Measurements per family:
- `mcc(M, hypernym)` — the hypothesis;
- `mcc(mean, hypernym)` — is the GDS common CLOSER to the hypernym than the
  plain sibling blend (which keeps difference structure too)?
- cross-family null: `mcc(M_i, hypernym_j)`, i != j;
- random-frame reference.


In [ ]:
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from nltk.corpus import wordnet2021 as wn

from frames.linalg.angles import mean_canonical_correlation
from frames.linalg.gds import gds_split
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
COMMON_THRESHOLD = 1.05
MIN_SIBLINGS = 3
N_FAMILIES = 15

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

OUT_CSV = Path("resources/35_hierarchy_alignment.csv")
OUT_PNG = Path("resources/35_hierarchy_alignment.png")

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)

df = fur.data.get_dataframe(fur.model.tokenizer, MIN_LEMMAS, MAX_TOKENS)
pdf = df.to_pandas() if hasattr(df, "to_pandas") else df
survivors = set(pdf["synset"].unique())
nouns = {s for s in survivors if ".n." in s}
print(f"{len(nouns)} surviving noun synsets")

families = defaultdict(list)
for name in sorted(nouns):
    try:
        s = wn.synset(name)
    except Exception:  # noqa: BLE001 - OMW names can be malformed
        continue
    for h in s.hypernyms():
        if h.name() in nouns:
            families[h.name()].append(name)

families = {h: kids for h, kids in families.items() if len(kids) >= MIN_SIBLINGS}
ranked = sorted(families.items(), key=lambda kv: -len(kv[1]))[:N_FAMILIES]
print(
    f"{len(families)} families with >= {MIN_SIBLINGS} siblings; using top {len(ranked)}:"
)
for h, kids in ranked:
    print(f"  {h}: {len(kids)} siblings, e.g. {[k.split('.')[0] for k in kids[:6]]}")

In [ ]:
def build(name):
    c = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    return Concept(synset=c.synset, tensor=c.tensor.to(fur.model.device))


rows, commons, hypernyms = [], {}, {}
for h_name, kids in ranked:
    try:
        hyper = build(h_name)
        siblings = [build(k) for k in kids]
    except Exception as error:  # noqa: BLE001 - survival is data-dependent
        print(f"  DROPPED family {h_name}: {type(error).__name__}")
        continue
    stack = torch.cat(Concept._pad_to_common_rank(siblings))
    common, _, eigenvalues = gds_split(stack, common_threshold=COMMON_THRESHOLD)
    if common.shape[1] == 0:
        print(f"  {h_name}: no common subspace — recorded as null-aligned")
        continue
    h_frame = hyper.tensor[0].double()
    mean_frame = Concept.average(siblings).tensor[0].double()
    commons[h_name] = common
    hypernyms[h_name] = h_frame
    rows.append(
        {
            "family": h_name,
            "n_siblings": len(siblings),
            "common_dim": int(common.shape[1]),
            "mcc_common_hyper": float(mean_canonical_correlation(common, h_frame)),
            "mcc_mean_hyper": float(mean_canonical_correlation(mean_frame, h_frame)),
            "max_eigenvalue": float(eigenvalues.max()),
        }
    )

results = pd.DataFrame(rows)
assert len(results) >= 5, "too few families survived to test the hypothesis"
print(results.round(3).to_string(index=False))

In [ ]:
# cross-family null: common subspace of family i vs hypernym of family j
null_vals = []
for i_name, common in commons.items():
    for j_name, h_frame in hypernyms.items():
        if i_name != j_name:
            null_vals.append(float(mean_canonical_correlation(common, h_frame)))
null = pd.Series(null_vals)

# random-frame reference at matched dimension
generator = torch.Generator(device="cpu").manual_seed(0)
d = next(iter(hypernyms.values())).shape[0]
rand_vals = []
for h_frame in hypernyms.values():
    gauss = torch.randn(d, 3, generator=generator, dtype=torch.float64)
    q, _ = torch.linalg.qr(gauss)
    rand_vals.append(float(mean_canonical_correlation(q.to(h_frame.device), h_frame)))
rand = pd.Series(rand_vals)

matched = results["mcc_common_hyper"]
print(f"matched  mcc: {matched.mean():.4f} +- {matched.std():.4f}")
print(f"null     mcc: {null.mean():.4f} +- {null.std():.4f}  (n={len(null)})")
print(f"random   mcc: {rand.mean():.4f} +- {rand.std():.4f}")
print(f"plain-mean vs hypernym: {results['mcc_mean_hyper'].mean():.4f}")
share_above = (matched > null.mean() + 2 * null.std()).mean()
print(f"\nfamilies with matched mcc > null + 2sd: {share_above:.0%}")
results.to_csv(OUT_CSV, index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
x = range(len(results))
axes[0].bar(x, results["mcc_common_hyper"], width=0.4, label="GDS common vs hypernym")
axes[0].bar(
    [i + 0.42 for i in x],
    results["mcc_mean_hyper"],
    width=0.4,
    label="plain mean vs hypernym",
)
axes[0].axhline(null.mean(), color="gray", ls="--", lw=0.9, label="cross-family null")
axes[0].axhline(rand.mean(), color="red", ls=":", lw=0.9, label="random frames")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(
    [f.split(".")[0] for f in results["family"]], rotation=45, fontsize=7, ha="right"
)
axes[0].set_ylabel("mean canonical correlation")
axes[0].legend(fontsize=7)

axes[1].hist(null, bins=25, alpha=0.6, label="cross-family null", color="gray")
for v in results["mcc_common_hyper"]:
    axes[1].axvline(v, color="tab:blue", lw=1.2, alpha=0.8)
axes[1].set_xlabel("mcc")
axes[1].set_ylabel("count")
axes[1].legend(["matched families (lines)", "cross-family null"], fontsize=7)

plt.suptitle("Branch C pilot: sibling common subspace vs WordNet hypernym frame")
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=150)
plt.show()